In [159]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import torchinfo
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from collections import Counter
from tqdm.auto import tqdm
import copy

import torchmetrics

import helper_utils


In [160]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA")
else:
    device = torch.device("cpu")
    print(f"Using device: CPU")

Using device: CUDA


## Hyperparameters

In [161]:
BATCH_SIZE = 32
LEARNING_RATE = 0.05
NUM_EPOCHS = 10

## Datasets and DataLoaders

### Dataset Transforms

In [162]:
train_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
])

val_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
])

### Train, Test and Validation Datasets and DataLoaders

In [163]:
train_dir = os.path.join("data", "train")
test_dir = os.path.join("data", "test")
val_dir = os.path.join("data", "validation")

train_dataset = ImageFolder(root = train_dir, transform = train_transforms)
val_dataset = ImageFolder(root = val_dir, transform = val_transforms)
test_dataset = ImageFolder(root = test_dir, transform = val_transforms)

train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = BATCH_SIZE, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = False)

In [164]:
classes = train_dataset.classes

num_classes = len(classes)

print(f"Classes: {classes}")
print(f"Number of classes: {num_classes}")

Classes: ['dress', 'hat', 'longsleeve', 'outwear', 'pants', 'shirt', 'shoes', 'shorts', 'skirt', 't-shirt']
Number of classes: 10


In [165]:
class InvertedResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride, expansion_factor, shortcut = None):
        super(InvertedResidualBlock, self).__init__()

        hidden_dim = in_channels * expansion_factor



        # 1 x 1 pointwise convolution to expand the number of channels
        self.expand = nn.Sequential(
            nn.Conv2d(in_channels =  in_channels,
                      out_channels = hidden_dim,
                      kernel_size = 1, 
                      bias = False
                      ),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace = True)
        )

        # 3 x 3 depthwise convolution
        self.depthwise = nn.Sequential(
            nn.Conv2d(in_channels = hidden_dim,
                      out_channels = hidden_dim,
                      kernel_size = 3,
                      stride = stride,
                      padding = 1,
                      bias = False,
                      groups = hidden_dim
                      ),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace = True)
        )

        # 1 x 1 pointwise convolution to project back to the original number of channels
        # Bottleneck
        self.project = nn.Sequential(
            nn.Conv2d(in_channels = hidden_dim,
                      out_channels = out_channels,
                      kernel_size = 1,
                      bias = False),
            nn.BatchNorm2d(out_channels)
        )

        # Optional shortcut connection for residual learning
        self.shortcut = shortcut



    def forward(self, x):

        skip = x

        out = self.expand(x)
        out = self.depthwise(out)
        out = self.project(out)
        
        if self.shortcut is not None:
            skip = self.shortcut(x)

        if skip.shape == out.shape:
            out = out + skip

        return F.relu(out)



In [166]:
class MobileNetBackbone(nn.Module):
    def __init__(self):

        super(MobileNetBackbone, self).__init__()


        self.stem = nn.Sequential(
           nn.Conv2d(in_channels = 3, out_channels = 16, kernel_size = 3,stride = 2, padding = 1, bias = False),
           nn.BatchNorm2d(16),
           nn.ReLU(inplace = True)
        )

        self.blocks = nn.Sequential(
            self._make_block(in_channels = 16, out_channels = 24, stride = 2, expansion_factor = 3),
            self._make_block(in_channels = 24, out_channels = 32, stride = 2, expansion_factor = 3),
            self._make_block(in_channels = 32, out_channels = 64, stride = 2, expansion_factor = 6),
            self._make_block(in_channels = 64, out_channels = 128, stride = 2, expansion_factor = 12)
        )





    def _make_block(self, in_channels, out_channels, stride, expansion_factor):

        condition = (stride !=1) or (in_channels != out_channels)

        if condition:
            shortcut = nn.Sequential(
                nn.Conv2d(in_channels = in_channels,
                          out_channels = out_channels,
                          kernel_size = 1,
                          stride = stride,
                          padding = 0,
                          bias = False),
                nn.BatchNorm2d(out_channels)
            )

        else:
            shortcut = None


        block = InvertedResidualBlock(
            in_channels = in_channels,
            out_channels = out_channels,
            stride = stride,
            expansion_factor = expansion_factor,
            shortcut = shortcut
        )

        return block

    def forward(self, x):
        x = self.stem(x)

        x = self.blocks(x)

        return x


In [167]:
# --- Verification ---
# Define parameters for verification
batch_size=32
img_size = 64 # Input image height/width
depth = 3 # Summary depth

# Instantiate the backbone
backbone = MobileNetBackbone()

# Define the input tensor shape
input_size = (batch_size, 3, img_size, img_size)

# Configuration for torchinfo summary
config = {
    "input_size": input_size,
    "attr_names": ["input_size", "output_size", "num_params"],
    "col_names_display": ["Input Shape ", "Output Shape", "Param #"],
    "depth": depth
}

# Generate the summary
summary = torchinfo.summary(
    model=backbone,
    input_size=config["input_size"],
    col_names=config["attr_names"],
    depth=config["depth"]
)

# Display the formatted summary
print("--- Backbone Summary ---\n")
helper_utils.display_torch_summary(summary, config["attr_names"], config["col_names_display"], config["depth"])

--- Backbone Summary ---



Layer (type (var_name):depth-idx),Input Shape,Output Shape,Param #
MobileNetBackbone (MobileNetBackbone),"[32, 3, 64, 64]","[32, 128, 2, 2]",--
Sequential (stem): 1-1,"[32, 3, 64, 64]","[32, 16, 32, 32]",--
Conv2d (0): 2-1,"[32, 3, 64, 64]","[32, 16, 32, 32]",432
BatchNorm2d (1): 2-2,"[32, 16, 32, 32]","[32, 16, 32, 32]",32
ReLU (2): 2-3,"[32, 16, 32, 32]","[32, 16, 32, 32]",--
Sequential (blocks): 1-2,"[32, 16, 32, 32]","[32, 128, 2, 2]",--
InvertedResidualBlock (0): 2-4,"[32, 16, 32, 32]","[32, 24, 16, 16]",--
Sequential (expand): 3-1,"[32, 16, 32, 32]","[32, 48, 32, 32]",864
Sequential (depthwise): 3-2,"[32, 48, 32, 32]","[32, 48, 16, 16]",528
Sequential (project): 3-3,"[32, 48, 16, 16]","[32, 24, 16, 16]","1,200"


In [168]:
class MobileNetLikeClassifier(nn.Module):
    def __init__(self, num_classes):

        super(MobileNetLikeClassifier, self).__init__()

        self.backbone = MobileNetBackbone()

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, num_classes)
        )


    def forward(self, x):
        x = self.backbone(x)

        x = self.head(x)

        return x

In [169]:
mobilenet_classifier = MobileNetLikeClassifier(num_classes=num_classes)

In [170]:
# Define parameters for verification
batch_size=32
img_size = 64 # Input image height/width
depth = 3 # Summary depth   

# Define the input tensor shape
input_size = (batch_size, 3, img_size, img_size)

# Configuration for torchinfo summary
config = {
    "input_size": input_size,
    "attr_names": ["input_size", "output_size", "num_params"],
    "col_names_display": ["Input Shape ", "Output Shape", "Param #"],
    "depth": depth # Show layers up to 3 levels deep for detail
}

# Generate the summary for the complete classifier
summary = torchinfo.summary(
    model=mobilenet_classifier,
    input_size=config["input_size"],
    col_names=config["attr_names"],
    depth=config["depth"]
)

# Display the formatted summary
print("--- Classifier Summary ---\n")
helper_utils.display_torch_summary(summary, config["attr_names"], config["col_names_display"], config["depth"])

--- Classifier Summary ---



Layer (type (var_name):depth-idx),Input Shape,Output Shape,Param #
MobileNetLikeClassifier (MobileNetLikeClassifier),"[32, 3, 64, 64]","[32, 10]",--
MobileNetBackbone (backbone): 1-1,"[32, 3, 64, 64]","[32, 128, 2, 2]",--
Sequential (stem): 2-1,"[32, 3, 64, 64]","[32, 16, 32, 32]",--
Conv2d (0): 3-1,"[32, 3, 64, 64]","[32, 16, 32, 32]",432
BatchNorm2d (1): 3-2,"[32, 16, 32, 32]","[32, 16, 32, 32]",32
ReLU (2): 3-3,"[32, 16, 32, 32]","[32, 16, 32, 32]",--
Sequential (blocks): 2-2,"[32, 16, 32, 32]","[32, 128, 2, 2]",--
InvertedResidualBlock (0): 3-4,"[32, 16, 32, 32]","[32, 24, 16, 16]","3,024"
InvertedResidualBlock (1): 3-5,"[32, 24, 16, 16]","[32, 32, 8, 8]","5,864"
InvertedResidualBlock (2): 3-6,"[32, 32, 8, 8]","[32, 64, 4, 4]","23,232"


In [171]:
# Function to compute class weights for the dataset
# Class weights will be used to handle data imbalance

def compute_class_weight(dataset):
    labels = dataset.targets


    class_counts = Counter(labels)

    # Class counts need to be sorted by class index because Counter does not guarantee order in terms of classes
    sorted_counts = [class_counts[i] for i in sorted(class_counts)]

    total_samples = len(labels)

    num_classes = len(class_counts)


    class_weights = []
    
    for count in sorted_counts:
        weight = total_samples / (num_classes * count)
        class_weights.append(weight)

    weights_tensor = torch.tensor(class_weights, dtype = torch.float)

    return weights_tensor

class_weights = compute_class_weight(train_dataset)
class_weights = class_weights.to(device)

In [172]:
loss_function = nn.CrossEntropyLoss(weight = class_weights)

# Print computed class weights
for i, weight in enumerate(class_weights):

    print(f'Class {classes[i]} weight: {weight}')

Class dress weight: 1.273029088973999
Class hat weight: 2.4943089485168457
Class longsleeve weight: 0.6742857098579407
Class outwear weight: 1.667391300201416
Class pants weight: 0.6555555462837219
Class shirt weight: 1.0579310655593872
Class shoes weight: 1.549494981765747
Class shorts weight: 1.518811821937561
Class skirt weight: 2.739285707473755
Class t-shirt weight: 0.3859119415283203


In [173]:
optimizer = torch.optim.Adam(mobilenet_classifier.parameters(),lr = LEARNING_RATE)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer = optimizer, step_size = 5, gamma = 0.1)

## Training Loop

In [174]:
mobilenet_classifier.to(device)
num_classes = len(classes)


# Training and Validation Accuracy Metrics
train_acc_metric = torchmetrics.Accuracy(task = 'multiclass', num_classes = num_classes).to(device)
val_acc_metric = torchmetrics.Accuracy(task = 'multiclass', num_classes = num_classes).to(device)

best_val_acc = 0.0

best_epoch = 0

model_best_state = None

for epoch in range(NUM_EPOCHS):

    mobilenet_classifier.train()

    running_train_loss = 0.0

    train_acc_metric.reset()

    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [TRAIN]", leave=False)

    for i, data in enumerate(train_pbar):

        inputs, labels = data

        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = mobilenet_classifier(inputs)

        loss = loss_function(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)

        preds = torch.argmax(outputs, 1)

        train_acc_metric.update(preds, labels)

        train_pbar.set_postfix(batch_loss=loss.item())

    epoch_train_loss = running_train_loss / len(train_loader.dataset)

    epoch_train_acc = train_acc_metric.compute().item()



    mobilenet_classifier.eval()

    running_val_loss = 0.0

    val_acc_metric.reset()

    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [VAL]", leave=False)

    with torch.no_grad():

        for images, labels in val_pbar:

            images, labels = images.to(device), labels.to(device)

            outputs = mobilenet_classifier(images)

            loss = loss_function(outputs, labels)

            running_val_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, 1)

            val_acc_metric.update(preds, labels)

            val_pbar.set_postfix(batch_loss=loss.item())

        epoch_val_loss = running_val_loss / len(val_loader.dataset)

        epoch_val_acc = val_acc_metric.compute().item()


        # Step the scheduler
        if scheduler is not None:
            # Check if it's ReduceLROnPlateau, which needs a metric
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                # Step the scheduler based on validation loss
                scheduler.step(epoch_val_loss)
            else:
                # Step other types of schedulers
                scheduler.step()
        
        # Log metrics for the epoch
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
              f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")
        
        # --- Check for Best Model ---
        # If current validation accuracy is better than the best seen so far
        if epoch_val_acc > best_val_acc:
            # Update the best validation accuracy
            best_val_acc = epoch_val_acc
            # Store the current epoch number
            best_epoch = epoch + 1
            # Use copy.deepcopy to save a snapshot of the model's state
            best_model_state = copy.deepcopy(mobilenet_classifier.state_dict())
            # Print a message indicating a new best model
            print(f"    ^ New best model found!")
        # --- End Check ---

    print("\n--- Training Complete ---")
    
    # --- Load Best Model ---
    # Check if a best model state was saved
    if best_model_state is not None:
        print(f"\nReturning best model from epoch {best_epoch} with {best_val_acc:.4f} validation accuracy.")
        # Load the best performing weights back into the model
        mobilenet_classifier.load_state_dict(best_model_state)
    else:
        # Warn if no improvement was seen and the last model is being returned
        print("\nWarning: No best model state was saved (e.g., validation never improved). Returning last model.")

Epoch 1/10 [TRAIN]:   0%|          | 0/96 [00:00<?, ?it/s]

Epoch 1/10 [VAL]:   0%|          | 0/11 [00:00<?, ?it/s]

Epoch 1/10 | Train Loss: 2.3656, Train Acc: 0.1408 | Val Loss: 2.2285, Val Acc: 0.1848
    ^ New best model found!

--- Training Complete ---

Returning best model from epoch 1 with 0.1848 validation accuracy.


Epoch 2/10 [TRAIN]:   0%|          | 0/96 [00:00<?, ?it/s]

Epoch 2/10 [VAL]:   0%|          | 0/11 [00:00<?, ?it/s]

...

Epoch 9/10 [VAL]:   0%|          | 0/11 [00:00<?, ?it/s]

Epoch 9/10 | Train Loss: 1.2818, Train Acc: 0.5486 | Val Loss: 1.3539, Val Acc: 0.5367
    ^ New best model found!

--- Training Complete ---

Returning best model from epoch 9 with 0.5367 validation accuracy.


Epoch 10/10 [TRAIN]:   0%|          | 0/96 [00:00<?, ?it/s]

Epoch 10/10 [VAL]:   0%|          | 0/11 [00:00<?, ?it/s]

Epoch 10/10 | Train Loss: 1.2121, Train Acc: 0.5847 | Val Loss: 1.3219, Val Acc: 0.5806
    ^ New best model found!

--- Training Complete ---

Returning best model from epoch 10 with 0.5806 validation accuracy.
